# Chatforge Storage Testing Guide (v2)

**Purpose**: Test and explore chatforge's participant-centric storage system.

**Schema v2 Features**:
- **Participants**: Humans and AIs are both participants in a chat
- **Multi-party chats**: Multiple users and/or AIs in one conversation
- **Sender snapshots**: Historical accuracy with `sender_name`
- **Attachments**: File upload support
- **Feedback**: Thumbs up/down and text feedback on messages
- **Tool tracking**: Track tool calls and agent runs

**Tables**:
1. `chats` - Conversation containers
2. `participants` - Who's in the chat (humans, AIs, bots)
3. `messages` - What was said
4. `attachments` - File attachments
5. `tool_calls` - Tool invocations
6. `agent_runs` - Agent execution sessions

---
## 1. Environment Setup

In [1]:
# Environment Setup
import sys
import os

# Ensure chatforge is importable
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import chatforge
print(f"Chatforge imported from: {chatforge.__file__}")

Chatforge imported from: /Users/ns/Desktop/projects/chatforge/chatforge/__init__.py


In [2]:
# Install required dependencies
%pip install sqlalchemy aiosqlite --quiet


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


---
## 2. Schema Overview

```
chats
├── participants (who's in the chat)
│   └── Human users, AI assistants, agents, bots
├── messages (what they said)
│   ├── participant_id → who sent it
│   ├── sender_name → snapshot of name at send time
│   ├── role → LLM role (user, assistant, system, tool)
│   ├── message_type → user, generated, fixed, edited
│   ├── feedback → thumbs_up_count, thumbs_down_count, text_feedback
│   └── attachment_id → optional file
├── attachments (files)
├── tool_calls (tool invocations)
└── agent_runs (AI execution sessions)
```

In [3]:
# Import the new schema models
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker

from chatforge.adapters.storage.models import (
    Base, Chat, Participant, Message, 
    Attachment, ToolCall, AgentRun
)

print("Models imported:")
print(f"  - Chat: {Chat.__tablename__}")
print(f"  - Participant: {Participant.__tablename__}")
print(f"  - Message: {Message.__tablename__}")
print(f"  - Attachment: {Attachment.__tablename__}")
print(f"  - ToolCall: {ToolCall.__tablename__}")
print(f"  - AgentRun: {AgentRun.__tablename__}")

Models imported:
  - Chat: chats
  - Participant: participants
  - Message: messages
  - Attachment: attachments
  - ToolCall: tool_calls
  - AgentRun: agent_runs


---
## 3. Create Database

In [4]:
import tempfile
import os

# Create a test database
db_path = os.path.join(tempfile.gettempdir(), "chatforge_v2_test.db")

# Remove if exists (fresh start)
if os.path.exists(db_path):
    os.remove(db_path)

engine = create_engine(f"sqlite:///{db_path}", echo=False)
Session = sessionmaker(bind=engine)

# Create all tables
Base.metadata.create_all(engine)

print(f"Database created: {db_path}")
print(f"Tables: {list(Base.metadata.tables.keys())}")

Database created: /var/folders/3w/9l9mlgsx74xgqtr3v7hy_1kr0000gn/T/chatforge_v2_test.db
Tables: ['chats', 'participants', 'messages', 'attachments', 'tool_calls', 'agent_runs']


---
## 4. Create a Chat with Participants

Key concept: **Both humans and AIs are participants**

In [5]:
from datetime import datetime, timezone

with Session() as session:
    # Create a chat
    chat = Chat(
        title="Test Conversation",
        settings={"model": "gpt-4o", "temperature": 0.7}
    )
    session.add(chat)
    session.flush()  # Get the ID
    
    # Add human participant
    human = Participant(
        chat_id=chat.id,
        participant_type="user",
        external_id="user-alice-123",  # Reference to host app's user
        display_name="Alice",
        role_in_chat="owner"
    )
    
    # Add AI participant
    ai = Participant(
        chat_id=chat.id,
        participant_type="assistant",
        external_id="gpt-4o",  # Model identifier
        display_name="Assistant",
        role_in_chat="member",
        metadata_={"model": "gpt-4o", "system_prompt": "You are a helpful assistant."}
    )
    
    session.add_all([human, ai])
    session.commit()
    
    print(f"Created chat: {chat.title} (ID: {chat.id})")
    print(f"\nParticipants:")
    print(f"  1. {human.display_name} ({human.participant_type}) - {human.role_in_chat}")
    print(f"  2. {ai.display_name} ({ai.participant_type}) - {ai.role_in_chat}")
    
    # Store IDs for later cells
    chat_id = chat.id
    human_id = human.id
    ai_id = ai.id

Created chat: Test Conversation (ID: 1)

Participants:
  1. Alice (user) - owner
  2. Assistant (assistant) - member


---
## 5. Send Messages

Key concept: **`sender_name` is a snapshot** - captures the name at send time

In [8]:
with Session() as session:
    # Get participants
    human = session.query(Participant).get(human_id)
    ai = session.query(Participant).get(ai_id)
    
    # User message
    msg1 = Message(
        chat_id=chat_id,
        participant_id=human.id,
        sender_name=human.display_name,  # Snapshot!
        role="user",
        message_type="user",
        content="Hello! What's 25 multiplied by 17?"
    )
    
    # AI response
    msg2 = Message(
        chat_id=chat_id,
        participant_id=ai.id,
        sender_name=ai.display_name,  # Snapshot!
        role="assistant",
        message_type="generated",
        content="25 multiplied by 17 equals 425.",
        token_count=45
    )
    
    # User follow-up
    msg3 = Message(
        chat_id=chat_id,
        participant_id=human.id,
        sender_name=human.display_name,
        role="user",
        message_type="user",
        content="Thanks! Can you explain how you calculated that?"
    )
    
    # AI explanation
    msg4 = Message(
        chat_id=chat_id,
        participant_id=ai.id,
        sender_name=ai.display_name,
        role="assistant",
        message_type="generated",
        content="I calculated 25 x 17 by breaking it down: 25 x 10 = 250, and 25 x 7 = 175. Then 250 + 175 = 425.",
        token_count=78
    )
    
    session.add_all([msg1, msg2, msg3, msg4])
    session.commit()
    
    print("Messages saved!")
    print(f"  Total messages: 4")

Messages saved!
  Total messages: 4


/var/folders/3w/9l9mlgsx74xgqtr3v7hy_1kr0000gn/T/ipykernel_89694/1937636580.py:3: LegacyAPIWarning: The Query.get() method is considered legacy as of the 1.x series of SQLAlchemy and becomes a legacy construct in 2.0. The method is now available as Session.get() (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  human = session.query(Participant).get(human_id)
/var/folders/3w/9l9mlgsx74xgqtr3v7hy_1kr0000gn/T/ipykernel_89694/1937636580.py:4: LegacyAPIWarning: The Query.get() method is considered legacy as of the 1.x series of SQLAlchemy and becomes a legacy construct in 2.0. The method is now available as Session.get() (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  ai = session.query(Participant).get(ai_id)


---
## 6. Retrieve Conversation

In [9]:
with Session() as session:
    # Get all messages for the chat
    messages = session.query(Message).filter(
        Message.chat_id == chat_id,
        Message.deleted_at.is_(None)
    ).order_by(Message.created_at).all()
    
    print(f"Conversation ({len(messages)} messages):")
    print("=" * 60)
    
    for msg in messages:
        emoji = "👤" if msg.role == "user" else "🤖"
        print(f"\n{emoji} {msg.sender_name} [{msg.role}]:")
        print(f"   {msg.content}")
        if msg.token_count:
            print(f"   (tokens: {msg.token_count})")

Conversation (12 messages):

👤 Alice [user]:
   Hello! What's 25 multiplied by 17?

🤖 Assistant [assistant]:
   25 multiplied by 17 equals 425.
   (tokens: 45)

👤 Alice [user]:
   Thanks! Can you explain how you calculated that?

🤖 Assistant [assistant]:
   I calculated 25 x 17 by breaking it down: 25 x 10 = 250, and 25 x 7 = 175. Then 250 + 175 = 425.
   (tokens: 78)

👤 Alice [user]:
   Hello! What's 25 multiplied by 17?

🤖 Assistant [assistant]:
   25 multiplied by 17 equals 425.
   (tokens: 45)

👤 Alice [user]:
   Thanks! Can you explain how you calculated that?

🤖 Assistant [assistant]:
   I calculated 25 x 17 by breaking it down: 25 x 10 = 250, and 25 x 7 = 175. Then 250 + 175 = 425.
   (tokens: 78)

👤 Alice [user]:
   Hello! What's 25 multiplied by 17?

🤖 Assistant [assistant]:
   25 multiplied by 17 equals 425.
   (tokens: 45)

👤 Alice [user]:
   Thanks! Can you explain how you calculated that?

🤖 Assistant [assistant]:
   I calculated 25 x 17 by breaking it down: 25 x 10 = 250,

---
## 7. Snapshot Demonstration

When a participant changes their name, old messages keep the old name.

In [10]:
with Session() as session:
    # Change Alice's display name
    human = session.query(Participant).get(human_id)
    old_name = human.display_name
    human.display_name = "Alice Smith"  # Updated name
    session.commit()
    
    print(f"Participant name changed: {old_name} -> {human.display_name}")
    
    # Send a new message with the new name
    msg = Message(
        chat_id=chat_id,
        participant_id=human.id,
        sender_name=human.display_name,  # New name snapshot
        role="user",
        message_type="user",
        content="This is a message with my updated name."
    )
    session.add(msg)
    session.commit()
    
    # Now check the messages
    messages = session.query(Message).filter(
        Message.chat_id == chat_id
    ).order_by(Message.created_at).all()
    
    print(f"\nMessage history (notice different sender names):")
    print("-" * 40)
    for msg in messages:
        if msg.role == "user":
            print(f"  {msg.sender_name}: {msg.content[:40]}...")

Participant name changed: Alice -> Alice Smith

Message history (notice different sender names):
----------------------------------------
  Alice: Hello! What's 25 multiplied by 17?...
  Alice: Thanks! Can you explain how you calculat...
  Alice: Hello! What's 25 multiplied by 17?...
  Alice: Thanks! Can you explain how you calculat...
  Alice: Hello! What's 25 multiplied by 17?...
  Alice: Thanks! Can you explain how you calculat...
  Alice Smith: This is a message with my updated name....


/var/folders/3w/9l9mlgsx74xgqtr3v7hy_1kr0000gn/T/ipykernel_89694/4057311186.py:3: LegacyAPIWarning: The Query.get() method is considered legacy as of the 1.x series of SQLAlchemy and becomes a legacy construct in 2.0. The method is now available as Session.get() (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  human = session.query(Participant).get(human_id)


---
## 8. Message Feedback (Thumbs Up/Down)

In [11]:
with Session() as session:
    # Get the AI's explanation message (the good one)
    ai_messages = session.query(Message).filter(
        Message.chat_id == chat_id,
        Message.role == "assistant"
    ).all()
    
    # Give thumbs up to the explanation
    explanation_msg = ai_messages[1]  # The second AI message
    explanation_msg.thumbs_up_count = 1
    explanation_msg.text_feedback = "Great step-by-step explanation!"
    
    session.commit()
    
    print(f"Feedback added to message ID {explanation_msg.id}:")
    print(f"  Content: {explanation_msg.content[:50]}...")
    print(f"  Thumbs up: {explanation_msg.thumbs_up_count}")
    print(f"  Thumbs down: {explanation_msg.thumbs_down_count}")
    print(f"  Text feedback: {explanation_msg.text_feedback}")

Feedback added to message ID 4:
  Content: I calculated 25 x 17 by breaking it down: 25 x 10 ...
  Thumbs up: 1
  Thumbs down: 0
  Text feedback: Great step-by-step explanation!


---
## 9. Tool Calls Tracking

In [12]:
with Session() as session:
    # Get the AI participant
    ai = session.query(Participant).get(ai_id)
    
    # Simulate user asking something that requires a tool
    user_msg = Message(
        chat_id=chat_id,
        participant_id=human_id,
        sender_name="Alice Smith",
        role="user",
        message_type="user",
        content="What's the weather in Tokyo?"
    )
    session.add(user_msg)
    session.flush()
    
    # AI response that used a tool
    ai_msg = Message(
        chat_id=chat_id,
        participant_id=ai_id,
        sender_name="Assistant",
        role="assistant",
        message_type="generated",
        content="The current weather in Tokyo is 15C and partly cloudy."
    )
    session.add(ai_msg)
    session.flush()
    
    # Log the tool call
    tool_call = ToolCall(
        message_id=ai_msg.id,
        tool_name="weather_api",
        input_params={"city": "Tokyo", "units": "celsius"},
        output_data={"temp": 15, "condition": "partly cloudy"},
        status="success",
        execution_time_ms=230
    )
    session.add(tool_call)
    session.commit()
    
    print(f"Tool call logged:")
    print(f"  Tool: {tool_call.tool_name}")
    print(f"  Input: {tool_call.input_params}")
    print(f"  Output: {tool_call.output_data}")
    print(f"  Status: {tool_call.status}")
    print(f"  Duration: {tool_call.execution_time_ms}ms")

Tool call logged:
  Tool: weather_api
  Input: {'city': 'Tokyo', 'units': 'celsius'}
  Output: {'temp': 15, 'condition': 'partly cloudy'}
  Status: success
  Duration: 230ms


/var/folders/3w/9l9mlgsx74xgqtr3v7hy_1kr0000gn/T/ipykernel_89694/4107348280.py:3: LegacyAPIWarning: The Query.get() method is considered legacy as of the 1.x series of SQLAlchemy and becomes a legacy construct in 2.0. The method is now available as Session.get() (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  ai = session.query(Participant).get(ai_id)


---
## 10. Agent Runs Tracking

In [13]:
from decimal import Decimal

with Session() as session:
    # Simulate an agent run
    agent_run = AgentRun(
        chat_id=chat_id,
        agent_name="ReActAgent",
        agent_version="1.0.0",
        status="completed",
        input_data={"query": "What's the weather in Tokyo?"},
        final_result={"response": "The weather in Tokyo is 15C and partly cloudy."},
        total_steps=3,
        total_tool_calls=1,
        token_usage={
            "prompt_tokens": 150,
            "completion_tokens": 45,
            "total_tokens": 195
        },
        cost=Decimal("0.000585")  # $0.000585
    )
    agent_run.completed_at = datetime.now(timezone.utc)
    session.add(agent_run)
    session.commit()
    
    print(f"Agent run logged:")
    print(f"  Agent: {agent_run.agent_name} v{agent_run.agent_version}")
    print(f"  Status: {agent_run.status}")
    print(f"  Steps: {agent_run.total_steps}")
    print(f"  Tool calls: {agent_run.total_tool_calls}")
    print(f"  Tokens: {agent_run.token_usage}")
    print(f"  Cost: ${agent_run.cost}")

Agent run logged:
  Agent: ReActAgent v1.0.0
  Status: completed
  Steps: 3
  Tool calls: 1
  Tokens: {'prompt_tokens': 150, 'completion_tokens': 45, 'total_tokens': 195}
  Cost: $0.000585


---
## 11. Group Chat with Multiple Participants

In [14]:
with Session() as session:
    # Create a group chat
    group_chat = Chat(
        title="Project Discussion",
        metadata_={"chat_type": "group"}
    )
    session.add(group_chat)
    session.flush()
    
    # Add multiple participants
    alice = Participant(
        chat_id=group_chat.id,
        participant_type="user",
        external_id="user-alice",
        display_name="Alice",
        role_in_chat="owner"
    )
    
    bob = Participant(
        chat_id=group_chat.id,
        participant_type="user",
        external_id="user-bob",
        display_name="Bob",
        role_in_chat="member"
    )
    
    ai_assistant = Participant(
        chat_id=group_chat.id,
        participant_type="assistant",
        external_id="gpt-4o",
        display_name="AI Helper",
        role_in_chat="member"
    )
    
    session.add_all([alice, bob, ai_assistant])
    session.flush()
    
    # Add messages from different participants
    messages = [
        Message(
            chat_id=group_chat.id,
            participant_id=alice.id,
            sender_name=alice.display_name,
            role="user",
            message_type="user",
            content="Hey team, let's discuss the new feature."
        ),
        Message(
            chat_id=group_chat.id,
            participant_id=bob.id,
            sender_name=bob.display_name,
            role="user",
            message_type="user",
            content="Sounds good! I have some ideas."
        ),
        Message(
            chat_id=group_chat.id,
            participant_id=ai_assistant.id,
            sender_name=ai_assistant.display_name,
            role="assistant",
            message_type="generated",
            content="I can help summarize the discussion and track action items."
        ),
    ]
    session.add_all(messages)
    session.commit()
    
    print(f"Group chat created: {group_chat.title}")
    print(f"\nParticipants ({len([alice, bob, ai_assistant])}):")  
    for p in [alice, bob, ai_assistant]:
        emoji = "👤" if p.participant_type == "user" else "🤖"
        print(f"  {emoji} {p.display_name} ({p.participant_type})")
    
    print(f"\nMessages:")
    for msg in messages:
        print(f"  {msg.sender_name}: {msg.content}")

Group chat created: Project Discussion

Participants (3):
  👤 Alice (user)
  👤 Bob (user)
  🤖 AI Helper (assistant)

Messages:
  Alice: Hey team, let's discuss the new feature.
  Bob: Sounds good! I have some ideas.
  AI Helper: I can help summarize the discussion and track action items.


---
## 12. Multi-Agent Chat (AI-to-AI)

In [15]:
with Session() as session:
    # Create an AI workspace
    ai_chat = Chat(
        title="Research Analysis",
        metadata_={"chat_type": "ai_workspace"}
    )
    session.add(ai_chat)
    session.flush()
    
    # Multiple AI agents
    researcher = Participant(
        chat_id=ai_chat.id,
        participant_type="agent",
        external_id="researcher-agent",
        display_name="Researcher",
        role_in_chat="owner",
        metadata_={"model": "claude-3-opus", "tools": ["web_search", "arxiv"]}
    )
    
    critic = Participant(
        chat_id=ai_chat.id,
        participant_type="agent",
        external_id="critic-agent",
        display_name="Critic",
        role_in_chat="member",
        metadata_={"model": "gpt-4o", "role": "challenge assumptions"}
    )
    
    session.add_all([researcher, critic])
    session.flush()
    
    # AI-to-AI conversation
    messages = [
        Message(
            chat_id=ai_chat.id,
            participant_id=researcher.id,
            sender_name=researcher.display_name,
            role="assistant",
            message_type="generated",
            content="Based on my research, transformer models show 15% improvement in efficiency."
        ),
        Message(
            chat_id=ai_chat.id,
            participant_id=critic.id,
            sender_name=critic.display_name,
            role="assistant",
            message_type="generated",
            content="What's the sample size? 15% seems optimistic without more context."
        ),
        Message(
            chat_id=ai_chat.id,
            participant_id=researcher.id,
            sender_name=researcher.display_name,
            role="assistant",
            message_type="generated",
            content="Good point. The study used 1000 samples across 5 datasets. Let me find more studies."
        ),
    ]
    session.add_all(messages)
    session.commit()
    
    print(f"AI Workspace: {ai_chat.title}")
    print(f"\nAgents:")
    for p in [researcher, critic]:
        print(f"  🤖 {p.display_name}: {p.metadata_}")
    
    print(f"\nConversation:")
    for msg in messages:
        print(f"  [{msg.sender_name}]: {msg.content}")

AI Workspace: Research Analysis

Agents:
  🤖 Researcher: {'model': 'claude-3-opus', 'tools': ['web_search', 'arxiv']}
  🤖 Critic: {'model': 'gpt-4o', 'role': 'challenge assumptions'}

Conversation:
  [Researcher]: Based on my research, transformer models show 15% improvement in efficiency.
  [Critic]: What's the sample size? 15% seems optimistic without more context.
  [Researcher]: Good point. The study used 1000 samples across 5 datasets. Let me find more studies.


---
## 13. Attachments

In [16]:
with Session() as session:
    # First create a message
    msg_with_attachment = Message(
        chat_id=chat_id,
        participant_id=human_id,
        sender_name="Alice Smith",
        role="user",
        message_type="user",
        content="Here's the report we discussed."
    )
    session.add(msg_with_attachment)
    session.flush()
    
    # Create an attachment linked to the message
    attachment = Attachment(
        message_id=msg_with_attachment.id,
        filename="report.pdf",
        file_type="application/pdf",
        file_size=1024 * 500,  # 500 KB
        storage_path="/uploads/2025/01/report.pdf",
        metadata_={"pages": 10, "title": "Q4 Report"}
    )
    session.add(attachment)
    session.commit()
    
    print(f"Message with attachment:")
    print(f"  Content: {msg_with_attachment.content}")
    print(f"  Attachment: {attachment.filename}")
    print(f"  Type: {attachment.file_type}")
    print(f"  Size: {attachment.file_size / 1024:.1f} KB")
    print(f"  Path: {attachment.storage_path}")

Message with attachment:
  Content: Here's the report we discussed.
  Attachment: report.pdf
  Type: application/pdf
  Size: 500.0 KB
  Path: /uploads/2025/01/report.pdf


---
## 14. Querying Data

In [17]:
with Session() as session:
    # Get all chats
    all_chats = session.query(Chat).filter(Chat.deleted_at.is_(None)).all()
    
    print(f"All Chats ({len(all_chats)}):")
    print("=" * 60)
    
    for chat in all_chats:
        # Active participants are those who haven't left
        participants = session.query(Participant).filter(
            Participant.chat_id == chat.id,
            Participant.left_at.is_(None)
        ).all()
        
        msg_count = session.query(Message).filter(
            Message.chat_id == chat.id
        ).count()
        
        chat_type = chat.metadata_.get("chat_type", "direct") if chat.metadata_ else "direct"
        
        print(f"\n📝 {chat.title} (ID: {chat.id})")
        print(f"  Type: {chat_type}")
        print(f"  Participants: {len(participants)}")
        for p in participants:
            emoji = "👤" if p.participant_type == "user" else "🤖"
            print(f"    - {emoji} {p.display_name} ({p.participant_type})")
        print(f"  Messages: {msg_count}")

All Chats (3):

📝 Test Conversation (ID: 1)
  Type: direct
  Participants: 2
    - 👤 Alice Smith (user)
    - 🤖 Assistant (assistant)
  Messages: 16

📝 Project Discussion (ID: 2)
  Type: group
  Participants: 3
    - 👤 Alice (user)
    - 👤 Bob (user)
    - 🤖 AI Helper (assistant)
  Messages: 3

📝 Research Analysis (ID: 3)
  Type: ai_workspace
  Participants: 2
    - 🤖 Researcher (agent)
    - 🤖 Critic (agent)
  Messages: 3


In [18]:
# Find chats by external user ID (how host app would query)
with Session() as session:
    external_user_id = "user-alice-123"
    
    # Find all chats where this user participates (and hasn't left)
    user_chats = session.query(Chat).join(Participant).filter(
        Participant.external_id == external_user_id,
        Participant.left_at.is_(None),
        Chat.deleted_at.is_(None)
    ).all()
    
    print(f"Chats for user '{external_user_id}':")
    for chat in user_chats:
        chat_type = chat.metadata_.get("chat_type", "direct") if chat.metadata_ else "direct"
        print(f"  - {chat.title} ({chat_type})")

Chats for user 'user-alice-123':
  - Test Conversation (direct)


In [19]:
# Get messages with feedback
with Session() as session:
    messages_with_feedback = session.query(Message).filter(
        (Message.thumbs_up_count > 0) | 
        (Message.thumbs_down_count > 0) |
        (Message.text_feedback.isnot(None))
    ).all()
    
    print(f"Messages with feedback ({len(messages_with_feedback)}):")
    for msg in messages_with_feedback:
        print(f"\n  Message: {msg.content[:50]}...")
        print(f"  Thumbs up: {msg.thumbs_up_count}")
        print(f"  Thumbs down: {msg.thumbs_down_count}")
        if msg.text_feedback:
            print(f"  Feedback: {msg.text_feedback}")

Messages with feedback (1):

  Message: I calculated 25 x 17 by breaking it down: 25 x 10 ...
  Thumbs up: 1
  Thumbs down: 0
  Feedback: Great step-by-step explanation!


---
## 15. Soft Delete

In [20]:
with Session() as session:
    # Soft delete a message
    msg = session.query(Message).filter(Message.chat_id == chat_id).first()
    
    print(f"Before soft delete:")
    print(f"  Message: {msg.content[:40]}...")
    print(f"  deleted_at: {msg.deleted_at}")
    
    msg.deleted_at = datetime.now(timezone.utc)
    session.commit()
    
    print(f"\nAfter soft delete:")
    print(f"  deleted_at: {msg.deleted_at}")
    
    # Query excludes soft-deleted by default
    active_messages = session.query(Message).filter(
        Message.chat_id == chat_id,
        Message.deleted_at.is_(None)
    ).count()
    
    all_messages = session.query(Message).filter(
        Message.chat_id == chat_id
    ).count()
    
    print(f"\nActive messages: {active_messages}")
    print(f"All messages (including deleted): {all_messages}")

Before soft delete:
  Message: Hello! What's 25 multiplied by 17?...
  deleted_at: None

After soft delete:
  deleted_at: 2025-12-27 07:14:16.757815

Active messages: 15
All messages (including deleted): 16


---
## 16. Convert to LLM Format

In [21]:
with Session() as session:
    # Get active messages
    messages = session.query(Message).filter(
        Message.chat_id == chat_id,
        Message.deleted_at.is_(None)
    ).order_by(Message.created_at).limit(5).all()
    
    # Convert to LLM format
    llm_messages = [msg.to_llm_format() for msg in messages]
    
    print("LLM-ready message format:")
    print("-" * 40)
    for msg in llm_messages:
        print(msg)

LLM-ready message format:
----------------------------------------
{'role': 'assistant', 'content': '25 multiplied by 17 equals 425.'}
{'role': 'user', 'content': 'Thanks! Can you explain how you calculated that?'}
{'role': 'assistant', 'content': 'I calculated 25 x 17 by breaking it down: 25 x 10 = 250, and 25 x 7 = 175. Then 250 + 175 = 425.'}
{'role': 'user', 'content': "Hello! What's 25 multiplied by 17?"}
{'role': 'assistant', 'content': '25 multiplied by 17 equals 425.'}


---
## 17. Using StoragePort (Abstraction Layer)

All the above examples used raw SQLAlchemy ORM directly. In production, you'd use the `StoragePort` abstraction via `SQLAlchemyStorageAdapter`.

**Why use the port?**
- Decouples your code from the database implementation
- Allows swapping backends (SQLite, PostgreSQL, in-memory)
- Provides a cleaner async API
- Handles conversions between ORM models and dataclasses

In [ ]:
import asyncio
from chatforge.adapters.storage import SQLAlchemyStorageAdapter

# Create adapter using existing engine/session
adapter = SQLAlchemyStorageAdapter(engine, Session)

async def demo_storage_port():
    # Setup (creates tables if needed)
    await adapter.setup()
    
    # 1. Create a chat (no user_id - ownership via participants)
    chat = await adapter.create_chat(
        title="StoragePort Demo",
        system_prompt="You are a helpful assistant."
    )
    print(f"✅ Created chat: {chat.id} - {chat.title}")
    
    # 2. Add participants
    human = await adapter.add_participant(
        chat_id=chat.id,
        participant_type="user",
        display_name="Demo User",
        external_id="user-demo-456",
        role_in_chat="owner"
    )
    
    ai = await adapter.add_participant(
        chat_id=chat.id,
        participant_type="assistant",
        display_name="Claude",
        role_in_chat="member"
    )
    print(f"✅ Added participants: {human.display_name}, {ai.display_name}")
    
    # 3. Save messages (sender_name auto-populated if not provided)
    msg1 = await adapter.save_message(
        chat_id=chat.id,
        content="What is 2+2?",
        role="user",
        participant_id=human.id
        # sender_name will be auto-populated from participant
    )
    
    msg2 = await adapter.save_message(
        chat_id=chat.id,
        content="2+2 equals 4.",
        role="assistant",
        participant_id=ai.id,
        sender_name=ai.display_name,
        message_type="generated",
        token_count=12
    )
    print(f"✅ Saved messages")
    
    # 4. Add feedback
    await adapter.add_feedback(
        message_id=msg2.id,
        thumbs_up=True,
        text_feedback="Correct and concise!"
    )
    print(f"✅ Added feedback to message")
    
    # 5. Add attachment
    attachment = await adapter.add_attachment(
        message_id=msg1.id,
        filename="math_problem.png",
        file_type="image/png",
        file_size=1024
    )
    print(f"✅ Added attachment: {attachment.filename}")
    
    # 6. Retrieve conversation
    messages = await adapter.get_messages(chat.id)
    print(f"\n📝 Conversation ({len(messages)} messages):")
    for msg in messages:
        print(f"   [{msg.role}] {msg.sender_name}: {msg.content}")
    
    # 7. List chats for user
    chats = await adapter.list_chats(external_user_id="user-demo-456")
    print(f"\n📋 User's chats: {[c.title for c in chats]}")
    
    print("\n✅ StoragePort demo complete!")

# Run the async demo
asyncio.run(demo_storage_port())

### Comparison: Raw ORM vs StoragePort

| Aspect | Raw ORM | StoragePort |
|--------|---------|-------------|
| **Setup** | Manual session management | `await adapter.setup()` |
| **Creating chat** | `session.add(Chat(...))` | `await adapter.create_chat(...)` |
| **Transactions** | Manual commit/rollback | Handled internally |
| **Return type** | ORM model | Dataclass record |
| **Async** | Not natively async | Async interface |
| **Swappable** | Tied to SQLAlchemy | Any adapter works |

**When to use what:**
- **Raw ORM**: Complex queries, migrations, direct database access
- **StoragePort**: Application logic, clean interfaces, testability

In [22]:
# Close engine and optionally delete test database
engine.dispose()

import os
if os.path.exists(db_path):
    os.remove(db_path)
    print(f"Deleted test database: {db_path}")

print("Cleanup complete!")

Deleted test database: /var/folders/3w/9l9mlgsx74xgqtr3v7hy_1kr0000gn/T/chatforge_v2_test.db
Cleanup complete!


---
## Summary

This notebook covered the **v2 participant-centric schema**:

| Feature | How |
|---------|-----|
| **Multi-party chats** | Multiple participants per chat |
| **Humans & AIs as peers** | Both are participants with `participant_type` |
| **Historical accuracy** | `sender_name` snapshot at send time |
| **User feedback** | `thumbs_up_count`, `thumbs_down_count`, `text_feedback` |
| **Attachments** | `attachments` table with file metadata |
| **Tool tracking** | `tool_calls` table for observability |
| **Agent runs** | `agent_runs` for cost and performance tracking |
| **Soft deletes** | `deleted_at` for data recovery |
| **Host app integration** | `external_id` references without owning users |
| **StoragePort abstraction** | Clean async API via `SQLAlchemyStorageAdapter` |

**Key Tables**:
1. `chats` - Conversation containers
2. `participants` - Who's in the chat
3. `messages` - What was said
4. `attachments` - File uploads
5. `tool_calls` - Tool invocations
6. `agent_runs` - AI execution sessions

**Two Ways to Use**:
- **Raw ORM**: Direct SQLAlchemy for complex queries and migrations
- **StoragePort**: `SQLAlchemyStorageAdapter` for clean application logic